# **Assignment: Fine-Tuning BERT for POS Tagging & Chunking**

In [1]:
!pip install transformers[torch] datasets evaluate seqeval

IMPORTS

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np

## LOAD DATASET

In [3]:
raw_datasets = load_dataset("lhoestq/conll2003")
raw_datasets

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

## LABEL CODE

In [4]:
from datasets import load_dataset

raw_datasets = load_dataset("lhoestq/conll2003")

# define label list (official CoNLL-2003 labels)
label_list = [
    "O",
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC",
    "B-MISC", "I-MISC"
]

label_list

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

## TOKENIZER CODE

In [5]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

## ALIGNMENT CODE

In [6]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

## APPLY TOKENIZATION

In [7]:
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True
)

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

## MODEL CODE

In [8]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label={i: l for i, l in enumerate(label_list)},
    label2id={l: i for i, l in enumerate(label_list)}
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## TRAINING CODE

In [10]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

training_args = TrainingArguments(
    "ner-distilbert-assignment",
    eval_strategy="epoch",    # Modern strategy name
    save_strategy="epoch",
    learning_rate=2e-5,       # Mandatory Learning Rate
    per_device_train_batch_size=16,
    num_train_epochs=3,       # Mandatory Epochs
    weight_decay=0.01,
    load_best_model_at_end=True
)

data_collator = DataCollatorForTokenClassification(tokenizer)

## METRICS CODE

In [11]:
import evaluate
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

## TRAIN MODEL

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].shuffle(seed=42).select(range(1500)),
    eval_dataset=tokenized_datasets["validation"].select(range(300)),
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,0.374101,0.577391,0.621723,0.598738
2,No log,0.218292,0.720486,0.777154,0.747748
3,No log,0.185680,0.740484,0.801498,0.769784


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=282, training_loss=0.3032297986619016, metrics={'train_runtime': 48.3347, 'train_samples_per_second': 93.101, 'train_steps_per_second': 5.834, 'total_flos': 55965397016592.0, 'train_loss': 0.3032297986619016, 'epoch': 3.0})

## INFERENCE CODE

In [20]:
from transformers import pipeline

ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer)

sentence = "Ganesh Mundal at Nashik completing B-Tech in computer science"
result = ner_pipeline(sentence)

# Clean output
for entity in result:
    print(f"Word: {entity['word']}, Label: {entity['entity']}, Score: {round(entity['score'], 2)}")

Word: gan, Label: B-PER, Score: 0.949999988079071
Word: ##esh, Label: B-PER, Score: 0.8700000047683716
Word: mu, Label: I-PER, Score: 0.9200000166893005
Word: ##nda, Label: I-PER, Score: 0.7900000214576721
Word: ##l, Label: I-PER, Score: 0.6800000071525574
Word: nash, Label: B-ORG, Score: 0.49000000953674316
Word: ##ik, Label: I-ORG, Score: 0.28999999165534973


### Inference Observation

Due to WordPiece tokenization, some words are split into subwords (e.g., "Ganesh" → "gan", "##esh").  
To improve readability, aggregation strategy is used to merge subwords into complete entities.

## Comparison: POS Tagging vs Chunking

| Feature | POS Tagging | Chunking |
|--------|------------|----------|
| Level | Word-level | Phrase-level |
| Output | Grammatical tags | Entity groups |
| Difficulty | Easy | Medium |
| Example | Noun, Verb | Noun Phrase |

Chunking provides better semantic understanding compared to POS tagging.

## Challenges Faced

- Handling subword tokenization
- Aligning labels with tokens
- Managing -100 ignored labels

## Observations

- Transformer models perform well for sequence labeling
- Chunking gives more meaningful context than POS tagging